# Sentiment Analysis Using Embeddings Averaging

## 1- Training phase

#### 1- Words → integers

* Each word in your sentence is mapped to a unique index in your vocab.

* Example: "This movie was great" → [2, 3, 4, 5]

#### 2- Pass integers to the NN

* The first layer is nn.Embedding

* Converts each integer → a vector (initially random)

* Output: (seq_len, embed_dim) per sentence, (batch_size, seq_len, embed_dim) for a batch

#### 3- Average the embeddings (mean pooling)

* Turns the sequence of vectors → one vector per sentence

* This is the sentence representation

#### 4-Feed into a classifier (fully connected layer)

* Output: logits for positive/negative

#### 5- Compute loss with label

* Compare predicted label with true label (0/1) using CrossEntropyLoss

#### 6- Backpropagation updates everything

* The classifier weights

* The embedding vectors themselves → they adjust so positive words push sentence vectors toward the 

* “positive” side and negative words toward “negative”

## 2 Testing / inference phase
#### 1- Take a new sentence → tokenize → encode → pad → integer indices
* "I loved this movie!" → [5, 2, 3, 7, 0, 0, …]
#### 2- Pass integers to the embedding layer
* Uses the vectors learned during training
* Output: (seq_len, embed_dim)
#### 3- Average the embeddings → get sentence vector
* Feed to classifier → predict positive/negative

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from collections import Counter
import numpy as np

we load our dataset

In [2]:
dataset = pd.read_csv('NLPdata/IMDB Dataset.csv')

the we encode the labels to 0 and 1 for negative and positive reviews respectively.

In [3]:
dataset['label'] = dataset["sentiment"].map({
    "positive": 1,
    "negative": 0
})

In [4]:
dataset.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


we dont need the sentiment column anymore

In [5]:
dataset.drop(columns=["sentiment"], inplace=True)

In [6]:
dataset.head()

,review,label
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


now we need to split the dataset into training and testing sets. We can use the train_test_split function from sklearn for this purpose.

In [7]:
dataset_train, dataset_test = train_test_split(dataset, test_size=0.2, random_state=42)

we analyze the length of reviews to determine the maximum length for padding and truncating
even though the maximum length is 13704, we will set it to 200-500 to reduce computational cost and avoid overfitting
the majority of meaningful information in reviews is likely to be contained within the first few hundred words, and longer reviews may contain more noise than useful information

In [8]:
ls = [len(x) for x in dataset_train["review"].values]

In [10]:
print(np.mean(ls))

1308.413825


In [11]:
print("Max review length:", max(ls))

Max review length: 13704


this is a simple tokenizer that splits the review into words based on spaces. You can replace it with a more sophisticated tokenizer if needed.

In [12]:
def tokenize(text):
    return text.lower().split()

then we create a vocabulary of the most common words in the training set, and assign an index to each word. We also add a special token for padding words and unknown that will have the index 0 and 1 respectively.
even though we have a lot of words in the dataset, we will only use the top 10000 most common words to create our vocabulary. This is a common practice in NLP to reduce the size of the vocabulary and improve the performance of the model.
this function will count the frequency of each word in the training set, and then create a vocabulary of the most common words by ordering them by frequency and taking the top 10000. 

| Token   | Index | Purpose                   |
| ------- | ----- | ------------------------- |
| `<PAD>` | 0     | Padding shorter sequences |
| `<UNK>` | 1     | Unknown / unseen words    |


``` python
vocab = {
  "<PAD>": 0,
  "<UNK>": 1,
  "movie": 2,
  "good": 3,
  "bad": 4,
  ...
}

In [ ]:
from collections import Counter

c=Counter()

for text in dataset_train["review"]:  # or "text" depending on dataset
    c.update(tokenize(text))

vocab_size = 10000
vocab = {word: i+2 for i, (word, _) in enumerate(c.most_common(vocab_size))}

vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

Encoding is the process of converting text into a sequence of integers using the previously built vocabulary.

``` python
["this", "movie", "is", "good"] → [1, 2, 1, 3] #using the built vocab

In [ ]:
def encode(text,max_len=200):
    tokens = tokenize(text)
    encoded = [vocab.get(word, vocab["<UNK>"]) for word in tokens]
    """PyTorch expects all tensors in the batch to have the same shape
    Also for mean pooling over seq_len, each sentence must have a consistent sequence length, 
    otherwise the averaging is inconsistent"""
    if len(encoded) < max_len:
        encoded += [vocab["<PAD>"]] * (max_len - len(encoded)) 
    else:
        encoded = encoded[:max_len]
    return torch.tensor(encoded, dtype=torch.long)

now we encoded all the reviews in the dataset, then we can create a dataloader for training and testing

In [ ]:
dataset_train["encoded"] = dataset_train["review"].apply(encode)
dataset_test["encoded"] = dataset_test["review"].apply(encode)
# Stack into tensors
X_train = torch.stack(dataset_train["encoded"].to_list())
y_train = torch.tensor(dataset_train["label"].to_list(), dtype=torch.long)

X_test = torch.stack(dataset_test["encoded"].to_list())
y_test = torch.tensor(dataset_test["label"].to_list(), dtype=torch.long)

# DataLoaders
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

In [17]:
dataset_train.head()

,review,label,encoded
39087,That's what I kept asking myself during the ma...,0,"[tensor(188), tensor(48), tensor(9), tensor(70..."
30893,I did not watch the entire movie. I could not ...,0,"[tensor(9), tensor(113), tensor(21), tensor(10..."
45278,A touching love story reminiscent of In the M...,1,"[tensor(3), tensor(1569), tensor(120), tensor(..."
16398,This latter-day Fulci schlocker is a totally a...,0,"[tensor(10), tensor(1), tensor(4307), tensor(1..."
13653,"First of all, I firmly believe that Norwegian ...",0,"[tensor(88), tensor(5), tensor(378), tensor(9)..."


our sentiment analysis model will be a simple feedforward network that takes the average of the word embeddings as input and outputs a binary sentiment prediction.
for that it will need the vocab size +2 for the <PAD> and <UNK> tokens, and an embedding dimension of 128 meaning each word will be represented by a 128-dimensional vector.

In [18]:
vocab_size = len(vocab)   # number of unique words + special tokens
embed_dim = 128            # size of vector for each word

* vocab_size = 10002
* Batch size = 32
* Max length = 200
* Embedding dim = 128

Input:        (32, 500)

Embedding:    (32, 500, 128)

Mean Pooling: (32, 128)

Linear:       (32, 2)

Output:       (32, 2)

In [19]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Linear(embed_dim, 2)  # 2 classes: positive / negative

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        x = self.embedding(x)       # (batch_size, seq_len, embed_dim)
        x = x.mean(dim=1)           # average pooling → (batch_size, embed_dim)
        x = self.fc(x)              # (batch_size, 2)
        return x

vocab_size = 10,000

embed_dim = 128

→ matrix = (10000, 128)

each row -> vector for one word

There is no activation after fc, and that’s intentional.  


Linear → (Softmax inside loss)

In [20]:
model = SentimentModel(vocab_size, embed_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

training the model

In [ ]:
for epoch in range(3):  # 3 epochs for example
    model.train()
    total_loss = 0

    for x, y in train_loader:
        optimizer.zero_grad()
        output = model(x)             # forward
        loss = criterion(output, y)   # compute loss
        loss.backward()               # backprop
        optimizer.step()              # update weights

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


KeyboardInterrupt



evaluating the model on test set

In [22]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for review, label in test_loader:
        output = model(review)
        preds = torch.max(output, 1)[1]
        correct += (preds == label).sum().item()
        total += label.size(0)

In [23]:
accuracy = correct / total
print("Test accuracy:", accuracy*100, "%")

Test accuracy: 86.42999999999999 %


In [27]:
def classify_review(model, review):
    model.eval()
    with torch.no_grad():
        output = model(encode(review).unsqueeze(0))  # encode and add batch dimension
        _, predicted = torch.max(output, 1)
    return "positive review" if predicted.item() else "negative review"

In [28]:
classify_review(model, "i enjoyed this movie.")

'positive review'

### Advantages
#### 1. Simplicity and Ease of Implementation

* Embedding → Mean Pooling → Linear
* Easy to understand, debug, and extend
* Requires minimal code compared to LSTM/CNN models

#### 2. Fast Training and Inference

* No sequential processing (like LSTM)
* No complex operations (like CNN filters)
* Works efficiently even on CPU

### Limitations

#### 1.  No Understanding of Word Order

Mean pooling ignores sequence information

* Example:

"this movie is good"  
"good movie is this"

 Both produce almost identical representations

#### 2.  Cannot Capture Context or Dependencies

Fails to understand relationships between words

* Example:

"not good"

 Model may interpret it similarly to:

"good"

#### 3.  Linear Model Limitation

Without hidden layers, the model is essentially:

Logistic Regression on averaged embeddings
Cannot learn complex patterns or nonlinear relationships

#### 4.  Information Loss from Averaging

Important words can be diluted during mean pooling

Long reviews lose strong signals


#### 5.  Limited Expressiveness Compared to Advanced Models
Cannot compete with:
* LSTM (captures sequence)
* CNN (captures local patterns)
* Transformers (captures full context)